In [1]:
#import the train data

import pandas as pd
train_df = pd.read_csv('C:/Users/3939/Documents/MMA Courses/MMA 867/X_train_mice_imputed_no_scaler2.csv')

train_df.head()

,LOAN,MORTDUE,VALUE,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC,...,JOB_Office,JOB_Other,JOB_ProfExe,JOB_Sales,JOB_Self,JOB_Unknown,REASON_HomeImp,REASON_Unknown,BAD,LTV
0,20000.0,77900.0,105400.0,4.000000,0.000000,0.0,110.600000,0.0,17.0,34.336862,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,27060.333761
1,23100.0,133867.0,174105.0,26.000000,0.538874,3.0,332.409840,0.0,28.0,30.768638,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,30043.442372
2,5500.0,52897.0,65054.0,20.000000,0.000000,0.0,189.295318,0.0,19.0,20.654129,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0,6764.031987
3,28100.0,100420.0,207737.0,22.000000,0.000000,0.0,197.198122,0.0,19.0,26.372549,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0,58129.951205
4,8000.0,36000.0,115800.0,9.682797,0.000000,0.0,168.233333,3.0,30.0,34.658298,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1,25733.333333


In [2]:
#import test data

test_df = pd.read_csv('C:/Users/3939/Documents/MMA Courses/MMA 867/X_test_mice_imputed_no_scaler2.csv')

test_df.head()

,LOAN,MORTDUE,VALUE,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC,...,JOB_Office,JOB_Other,JOB_ProfExe,JOB_Sales,JOB_Self,JOB_Unknown,REASON_HomeImp,REASON_Unknown,BAD,LTV
0,30000.0,224000.0,282000.0,5.000000,0.0,1.000000,209.366667,0.000000,30.0,36.958713,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,37767.857143
1,23700.0,43712.0,75191.0,6.000000,0.0,0.000000,50.717535,2.000000,18.0,37.103251,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,40767.448298
2,20800.0,29503.0,54923.0,10.226464,0.0,0.000000,215.480714,0.000000,12.0,38.732132,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0,38721.431719
3,20200.0,35819.0,49843.0,4.000000,0.0,0.484313,121.102230,1.008802,11.0,27.037513,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,28108.785840
4,30000.0,17696.0,53100.0,20.000000,4.0,5.000000,147.866667,0.000000,28.0,35.341231,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,90020.343580


In [3]:
#Separate the target variable for the train data

y_train=train_df['BAD']
X_train=train_df.drop(columns=['BAD'])

In [4]:
#Separate the target variable for the test data

y_test=test_df['BAD']
X_test=test_df.drop(columns=['BAD'])

In [5]:
#KNN
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
import numpy as np


In [39]:
#create a knn model
knn = KNeighborsClassifier()

#create a dictionary of all values we want to test for n_neighbors
param_grid = {'n_neighbors': (2,3,4,5,6,7,8,9,10)}
#use gridsearch to test all values for n_neighbors
knn_gscv = GridSearchCV(knn, param_grid, cv=10)#fit model to data
knn_gscv.fit(X_train, y_train)

GridSearchCV(cv=10, estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': (2, 3, 4, 5, 6, 7, 8, 9, 10)})

In [40]:
knn_gscv.best_params_

{'n_neighbors': 2}

In [41]:
from dmba import classificationSummary

classificationSummary(y_train, knn_gscv.predict(X_train))
classificationSummary(y_test, knn_gscv.predict(X_test))

Confusion Matrix (Accuracy 0.9027)

       Prediction
Actual     0     1
     0 13766     2
     1  1616  1246
Confusion Matrix (Accuracy 0.8447)

       Prediction
Actual    0    1
     0 5733  208
     1  899  288


The accuracy of the train data is higher than that of the test data. This may be an indicator of overfitting the train datasets. The precision and recall on both the train and test datasets will be calculated to further evaluate. 

In [46]:
# calcuate precision and recall on the train data
tp = 1246
fp = 2
tn = 13766
fn = 1616
prc = tp / (tp + fp)
rec = tp / (tp + fn)
print(f"Model's precision on trained data is {prc:.2f} and it's recall is {rec:.2f}")


Model's precision on trained data is 1.00 and it's recall is 0.44


In [47]:
# calcuate precision and recall on the test data
tp = 288
fp = 208
tn = 5733
fn = 899
prc = tp / (tp + fp)
rec = tp / (tp + fn)
print(f"Model's precision on test data is {prc:.2f} and it's recall is {rec:.2f}")


Model's precision on test data is 0.58 and it's recall is 0.24


The contrasting difference between the precision and recall of the trained and test datasets indicates an overfitting of the trained dataset. GridSearchCV will be run again to find the best parameter

In [50]:
#create a knn model
knn = KNeighborsClassifier()

#create a dictionary of all values we want to test for n_neighbors
param_grid = {'n_neighbors': np.arange(1, 25)}
#use gridsearch to test all values for n_neighbors
knn_gscv = GridSearchCV(knn, param_grid, cv=10)#fit model to data
knn_gscv.fit(X_train, y_train)

GridSearchCV(cv=10, estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24])})

In [51]:
knn_gscv.best_params_

{'n_neighbors': 24}

In [52]:
#create a knn model
knn = KNeighborsClassifier()

#create a dictionary of all values we want to test for n_neighbors
param_grid = {'n_neighbors': (21,22,23,24,25,26,27,28,29,30)}
#use gridsearch to test all values for n_neighbors
knn_gscv = GridSearchCV(knn, param_grid, cv=10)#fit model to data
knn_gscv.fit(X1_train, y1_train)

GridSearchCV(cv=10, estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': (21, 22, 23, 24, 25, 26, 27, 28, 29,
                                         30)})

In [53]:
knn_gscv.best_params_

{'n_neighbors': 27}

In [57]:
from dmba import classificationSummary

classificationSummary(y1_train, knn_gscv.predict(X1_train))
classificationSummary(y1_test, knn_gscv.predict(X1_test))

Confusion Matrix (Accuracy 0.8480)

       Prediction
Actual     0     1
     0 13552   216
     1  2312   550
Confusion Matrix (Accuracy 0.8485)

       Prediction
Actual    0    1
     0 5831  110
     1  970  217


This similar accuracy results of both the train and test datasets indicates no overfitting. Further evaluation using Precision and Recall.

In [55]:
# calcuate precision and recall on the train data
tp = 550
fp = 216
tn = 13552
fn = 2312
prc = tp / (tp + fp)
rec = tp / (tp + fn)
print(f"Model's precision on trained data is {prc:.2f} and it's recall is {rec:.2f}")

Model's precision on trained data is 0.72 and it's recall is 0.19


In [56]:
# calcuate precision and recall on the test data
tp = 217
fp = 110
tn = 5831
fn = 970
prc = tp / (tp + fp)
rec = tp / (tp + fn)
print(f"Model's precision on test data is {prc:.2f} and it's recall is {rec:.2f}")


Model's precision on test data is 0.66 and it's recall is 0.18


The similarities between the scores of the Train and Test data indicates no overfitting in train datasets.



The model exhibits high precision but low recall, suggesting it is good at identifying defaults when it predicts them but struggles to identify a large portion of all actual defaults. This imbalance indicates that while the model is accurate when it makes a prediction, it may be failing to capture many default cases, potentially leading to a high number of false negatives.